# Taller de movilidad urbana: taxis y clima en Nueva York

**Caso de estudio:** viajes de NYC TLC Yellow Taxi durante enero de 2024.

En este taller construiremos un flujo reproducible: adquisición, inspección, auditoría de calidad, enriquecimiento por zonas, agregación temporal, integración con clima y análisis exploratorio. No buscamos demostrar causalidad, sino aprender a formular y comprobar preguntas con datos reales.

## Objetivos

Al finalizar podrás:

- construir y validar una URL de datos abiertos;
- inspeccionar estructura, tipos y faltantes antes de analizar;
- convertir reglas de calidad en una tabla de auditoría;
- enriquecer viajes mediante uniones `many_to_one`;
- agregar pickups por zona y hora sin sesgar la serie temporal;
- convertir observaciones meteorológicas de UTC a `America/New_York`;
- comunicar patrones y límites de un análisis exploratorio.

> **Pregunta guía:** ¿cómo varían los pickups de Yellow Taxi por lugar y hora, y qué relación exploratoria muestran con temperatura y precipitación?

## Ficha de fuentes

| Fuente | Recurso usado | Papel en el taller |
|---|---|---|
| [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) | Yellow Taxi, enero de 2024 (`yellow_tripdata_2024-01.parquet`) | fecha/hora, zonas, distancia, pasajeros e importes de viajes reportados |
| NYC TLC | [Taxi Zone Lookup](https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv) | traduce `LocationID` a borough, zona y `service_zone` |
| NYC TLC | [Taxi Zone Shapefile](https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip) | geometrías para la extensión espacial opcional |
| [NOAA GHCNh](https://www.ncei.noaa.gov/products/global-historical-climatology-network-hourly) | estación `USW00094728`, año 2024 | observaciones de clima con marcas temporales UTC |

**Periodo común:** enero de 2024 en hora local de Nueva York. Los archivos se leen desde sus URL o desde un directorio temporal; este notebook no guarda datasets en el repositorio. Consulta la documentación y licencias de cada proveedor antes de reutilizar los datos.

## 1. Preparación y modo de ejecución

En Colab normalmente basta con ejecutar los imports. Si falta el motor Parquet, descomenta la instalación. La extensión espacial se instala solo si se desea ejecutar esa sección.

In [ ]:
# Instalación opcional para Google Colab/Jupyter (descomentar si hace falta):
# %pip install -q pyarrow
# %pip install -q geopandas folium requests

from io import StringIO
import tempfile
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# True: primera semana de enero; False: todo enero. No hay muestreo aleatorio.
MODO_CLASE = False
INICIO_MES = pd.Timestamp("2024-01-01")
FIN_MES = pd.Timestamp("2024-02-01")
FIN_ANALISIS = pd.Timestamp("2024-01-08") if MODO_CLASE else FIN_MES
ZONA_HORARIA = "America/New_York"

print(f"Ventana: {INICIO_MES} hasta {FIN_ANALISIS} (fin no incluido)")

`MODO_CLASE` limita la lectura mediante un filtro temporal de Parquet cuando el motor lo permite. La selección es una semana completa y determinista, no una muestra aleatoria: así preservamos horas consecutivas y el flujo de agregación. En modo completo, la misma lógica procesa todo enero.

## 2. URL validada y carga reproducible

In [ ]:
def construir_url_tlc(tipo, anio, mes):
    """Construye una URL TLC válida para un tipo, año y mes."""
    prefijos = {
        "yellow": "yellow_tripdata",
        "green": "green_tripdata",
        "fhv": "fhv_tripdata",
        "fhvhv": "fhvhv_tripdata",
    }
    if tipo not in prefijos:
        opciones = ", ".join(prefijos)
        raise ValueError(f"tipo debe ser uno de: {opciones}")
    if isinstance(anio, bool) or not isinstance(anio, (int, np.integer)) or not 2009 <= anio <= 2100:
        raise ValueError("anio debe ser un entero entre 2009 y 2100")
    if isinstance(mes, bool) or not isinstance(mes, (int, np.integer)) or not 1 <= mes <= 12:
        raise ValueError("mes debe ser un entero entre 1 y 12")
    archivo = f"{prefijos[tipo]}_{anio}-{mes:02d}.parquet"
    return f"https://d37ci6vzurychx.cloudfront.net/trip-data/{archivo}"

URL_VIAJES = construir_url_tlc("yellow", 2024, 1)
URL_ZONAS = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
URL_GEOMETRIAS = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"
URL_CLIMA = (
    "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/"
    "hourly/access/by-year/2024/psv/GHCNh_USW00094728_2024.psv"
)
print(URL_VIAJES)

In [ ]:
COLUMNAS_VIAJES = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "PULocationID",
    "DOLocationID", "fare_amount", "total_amount",
]
filtros = [
    ("tpep_pickup_datetime", ">=", INICIO_MES.to_pydatetime()),
    ("tpep_pickup_datetime", "<", FIN_ANALISIS.to_pydatetime()),
]

# PyArrow aplica el filtro por fecha al leer los grupos de filas compatibles.
viajes_originales = pd.read_parquet(
    URL_VIAJES, columns=COLUMNAS_VIAJES, filters=filtros, engine="pyarrow"
)
print(f"Viajes cargados: {len(viajes_originales):,}")

### Pregunta breve

¿Por qué una semana consecutiva es preferible a una muestra aleatoria si queremos comparar horas y días? Escribe una hipótesis antes de continuar.

<details><summary>Ver respuesta orientativa</summary>Una muestra aleatoria puede dejar horas incompletas y alterar sus conteos. Una ventana consecutiva conserva el orden, los ciclos diarios y el denominador temporal, aunque no necesariamente representa el mes completo.</details>

## 3. Inspección antes de limpiar

No debemos decidir reglas sin conocer el esquema. Observa dimensiones, ejemplos, tipos y faltantes.

In [ ]:
print("Shape:", viajes_originales.shape)
display(viajes_originales.head())
print("\nInfo:")
viajes_originales.info()
print("\nTipos:")
display(viajes_originales.dtypes.rename("dtype").to_frame())

faltantes = (
    viajes_originales.isna().sum().rename("n_faltantes").to_frame()
    .assign(porcentaje=lambda x: 100 * x["n_faltantes"] / len(viajes_originales))
    .sort_values("porcentaje", ascending=False)
)
display(faltantes)

**Observa:** `passenger_count` puede faltar por la forma de reporte. Su ausencia se conserva y no invalida automáticamente un viaje. Solo marcaremos como problemáticos los valores presentes fuera de un intervalo operativo razonable.

## 4. Calidad y auditoría

Las reglas siguientes son criterios analíticos, no verdades universales. Distancia máxima de 100 millas, hasta 8 pasajeros e importes menores a USD 1,000 son umbrales conservadores para detectar registros extremos o inválidos. Cada regla queda en una columna booleana.

In [ ]:
calidad = viajes_originales.copy()
calidad["duracion_min"] = (
    calidad["tpep_dropoff_datetime"] - calidad["tpep_pickup_datetime"]
).dt.total_seconds() / 60

calidad["ok_pickup"] = calidad["tpep_pickup_datetime"].notna() & calidad["tpep_pickup_datetime"].between(INICIO_MES, FIN_ANALISIS, inclusive="left")
calidad["ok_dropoff"] = calidad["tpep_dropoff_datetime"].notna()
calidad["ok_duracion"] = calidad["duracion_min"].gt(0) & calidad["duracion_min"].le(24 * 60)
calidad["ok_distancia"] = calidad["trip_distance"].between(0, 100, inclusive="both")
calidad["ok_id_pickup_presente"] = calidad["PULocationID"].notna()
calidad["ok_id_dropoff_presente"] = calidad["DOLocationID"].notna()
calidad["ok_importes"] = (
    calidad["fare_amount"].between(0, 1_000, inclusive="both")
    & calidad["total_amount"].between(0, 1_000, inclusive="both")
    & calidad["total_amount"].ge(calidad["fare_amount"])
)
calidad["ok_pasajeros"] = calidad["passenger_count"].isna() | calidad["passenger_count"].between(0, 8, inclusive="both")

REGLAS = [c for c in calidad.columns if c.startswith("ok_")]
calidad["registro_valido"] = calidad[REGLAS].all(axis=1)
auditoria = pd.DataFrame({
    "regla": REGLAS,
    "cumplen": [int(calidad[c].sum()) for c in REGLAS],
    "incumplen": [int((~calidad[c]).sum()) for c in REGLAS],
})
auditoria["porcentaje_incumple"] = 100 * auditoria["incumplen"] / len(calidad)
display(auditoria.sort_values("porcentaje_incumple", ascending=False))

In [ ]:
# Se mantienen por separado para no borrar silenciosamente información.
viajes_rechazados = calidad.loc[~calidad["registro_valido"]].copy()
viajes_validos = calidad.loc[calidad["registro_valido"]].copy()
print(f"Válidos: {len(viajes_validos):,} | Rechazados: {len(viajes_rechazados):,}")
display(viajes_rechazados[COLUMNAS_VIAJES + ["duracion_min"] + REGLAS].head())

### Ejercicio 1: sensibilidad de una regla

Calcula cuántos viajes quedarían fuera si la distancia máxima fuese 50 millas, sin modificar `viajes_validos`.

In [ ]:
# Tu respuesta:
# fuera_con_umbral_50 = ...

<details><summary>Ver solución</summary>

```python
fuera_con_umbral_50 = (~calidad["trip_distance"].between(0, 50)).sum()
print(fuera_con_umbral_50)
```

Conviene comparar este resultado con `ok_distancia` y justificar el umbral con conocimiento del dominio.
</details>

## 5. Enriquecimiento con zonas

### Objetivo

La tabla de viajes identifica el origen mediante `PULocationID` y el destino mediante `DOLocationID`. Estos códigos son útiles para relacionar tablas, pero no permiten interpretar directamente dónde comenzó o terminó un viaje.

En esta sección agregaremos a cada viaje el nombre de la zona, el borough y la clasificación de servicio tanto del origen como del destino. Conceptualmente:

```text
PULocationID = 161
        ↓
pickup_zone = Midtown Center
pickup_borough = Manhattan
pickup_service_zone = Yellow Zone
```

El enriquecimiento no cambia la unidad de observación: cada fila continúa representando un viaje Yellow Taxi reportado. Solo agrega contexto geográfico a sus identificadores.

### 5.1. El catálogo de zonas

`taxi_zone_lookup.csv` es un **catálogo de correspondencias** publicado por NYC TLC. No contiene viajes: contiene una descripción por `LocationID`.

| Columna | Significado |
|---|---|
| `LocationID` | Identificador numérico de la zona TLC |
| `Borough` | Distrito amplio, como Manhattan, Queens o Brooklyn |
| `Zone` | Nombre específico de la zona TLC |
| `service_zone` | Clasificación operativa, como `Yellow Zone`, `Boro Zone` o `Airports` |

Para que el catálogo pueda representar el lado **uno** de una unión, `LocationID` debe ser único.

In [ ]:
zonas = pd.read_csv(URL_ZONAS)
display(zonas.head())
print("Cantidad de zonas:", len(zonas))
print("LocationID es único:", zonas["LocationID"].is_unique)

if zonas["LocationID"].duplicated().any():
    raise ValueError("Taxi Zone Lookup contiene LocationID duplicados")

### 5.2. Un catálogo, dos roles

Cada viaje consulta el mismo catálogo desde dos papeles diferentes:

```text
PULocationID → zona de origen o pickup
DOLocationID → zona de destino o dropoff
```

Por eso creamos `zonas_pickup` y `zonas_dropoff`. No son dos catálogos distintos ni duplican los viajes: son dos versiones del mismo catálogo con nombres que explicitan el rol de sus columnas.

| Origen | Destino |
|---|---|
| `pickup_zone` | `dropoff_zone` |
| `pickup_borough` | `dropoff_borough` |
| `pickup_service_zone` | `dropoff_service_zone` |

Renombrar antes de unir evita nombres ambiguos como `Zone_x` y `Zone_y`. `DataFrame.rename()` devuelve estas versiones adaptadas sin modificar `zonas`.

In [ ]:
zonas_pickup = zonas.rename(columns={
    "LocationID": "PULocationID", "Borough": "pickup_borough",
    "Zone": "pickup_zone", "service_zone": "pickup_service_zone",
})
zonas_dropoff = zonas.rename(columns={
    "LocationID": "DOLocationID", "Borough": "dropoff_borough",
    "Zone": "dropoff_zone", "service_zone": "dropoff_service_zone",
})

display(zonas_pickup.head(2))
display(zonas_dropoff.head(2))

### 5.3. Cardinalidad `many_to_one`

La relación esperada es **muchos viajes → una descripción de zona**. Un mismo `PULocationID` o `DOLocationID` puede aparecer en miles de viajes, pero cada identificador debe aparecer como máximo una vez en el catálogo correspondiente.

`validate="many_to_one"` hace que pandas compruebe esta condición durante la unión. Si el catálogo tuviera dos filas para `LocationID = 161`, tres viajes con ese identificador producirían seis filas:

```text
3 viajes × 2 coincidencias en el catálogo = 6 filas
```

Esa multiplicación inflaría conteos, importes, promedios y mapas. La validación detiene el proceso con un error en lugar de producir silenciosamente un resultado incorrecto. También ayuda a detectar catálogos repetidos, mezcla de versiones o descripciones contradictorias para un mismo ID.

### 5.4. Dos uniones izquierdas

Primero agregamos los atributos del origen y después los del destino. `how="left"` conserva todos los viajes válidos. Si un ID no aparece en el catálogo, el viaje permanece y sus atributos geográficos quedan como `NaN`, lo que permite auditar la falta de correspondencia en vez de eliminarla silenciosamente.

In [ ]:
viajes_zonas = (
    viajes_validos.merge(zonas_pickup, on="PULocationID", how="left", validate="many_to_one")
    .merge(zonas_dropoff, on="DOLocationID", how="left", validate="many_to_one")
)

### 5.5. Controles posteriores e interpretación

Después de unir comprobamos dos propiedades diferentes:

1. **Conservación de filas:** la cantidad de viajes no debe cambiar.
2. **Cobertura del catálogo:** contamos los viajes cuyo ID no encontró nombre de zona.

`many_to_one` no garantiza cobertura completa ni detecta viajes duplicados, IDs faltantes, nombres incorrectos en una fila única o el uso de la clave equivocada. Por eso se complementa con estos controles y con la revisión semántica de las columnas obtenidas.

In [ ]:
assert len(viajes_zonas) == len(viajes_validos), "La unión cambió el número de viajes"

sin_nombre_pickup = viajes_zonas["pickup_zone"].isna().sum()
sin_nombre_dropoff = viajes_zonas["dropoff_zone"].isna().sum()
print("Viajes antes y después:", len(viajes_validos), len(viajes_zonas))
print("Sin zona pickup:", sin_nombre_pickup, "| Sin zona dropoff:", sin_nombre_dropoff)

COLUMNAS_ZONA = [
    "PULocationID", "pickup_borough", "pickup_zone", "pickup_service_zone",
    "DOLocationID", "dropoff_borough", "dropoff_zone", "dropoff_service_zone",
]
display(viajes_zonas[COLUMNAS_ZONA].head())

### Resultado de la sección

Cada fila sigue representando un viaje, pero ahora permite interpretar tanto su origen como su destino. Las zonas son áreas geográficas agregadas definidas por TLC: no indican una dirección ni una coordenada exacta, y el enriquecimiento no agrega demanda total, solicitudes no atendidas ni vehículos disponibles.

### Pregunta breve

¿Qué ocurriría si `LocationID = 161` apareciera dos veces en el catálogo y no utilizáramos `validate="many_to_one"`?

<details><summary>Ver respuesta orientativa</summary>Cada viaje con ese identificador encontraría dos coincidencias y aparecería dos veces en el resultado. Los conteos y agregaciones posteriores quedarían inflados. Con `validate="many_to_one"`, pandas detecta que el lado del catálogo no es único y detiene la unión.</details>

## 6. Agregación zona-hora de pickups

Los timestamps TLC se interpretan como hora local. En enero de 2024 no hay transición de horario de verano, pero explicitar la zona horaria evita una unión ambigua con NOAA.

In [ ]:
viajes_zonas["pickup_hora"] = (
    viajes_zonas["tpep_pickup_datetime"]
    .dt.tz_localize(ZONA_HORARIA, ambiguous="raise", nonexistent="raise")
    .dt.floor("h")
)
pickups_zona_hora = (
    viajes_zonas.groupby(
        ["pickup_hora", "PULocationID", "pickup_borough", "pickup_zone"],
        observed=True, dropna=False,
    )
    .size().rename("pickups").reset_index()
)
assert pickups_zona_hora.duplicated(["pickup_hora", "PULocationID"]).sum() == 0
display(pickups_zona_hora.head())

## 7. Clima: UTC, hora local y resumen horario

GHCNh puede contener varias observaciones dentro de una hora. Seleccionaremos variables numéricas, convertiremos valores no numéricos a faltantes y resumiremos a una fila por hora. Temperatura usa media; precipitación usa suma con `min_count=1` para no convertir una hora totalmente faltante en cero.

In [ ]:
COLUMNAS_CLIMA = [
    "STATION", "Station_name", "DATE", "temperature",
    "relative_humidity", "wind_speed", "precipitation", "visibility",
]
with urlopen(URL_CLIMA) as respuesta:
    texto_clima = respuesta.read().decode("utf-8")
clima_bruto = pd.read_csv(StringIO(texto_clima), sep="|", usecols=COLUMNAS_CLIMA, low_memory=False)
assert clima_bruto["STATION"].astype(str).eq("USW00094728").all()
print("Observaciones meteorológicas cargadas:", len(clima_bruto))

In [ ]:
clima = clima_bruto.copy()
clima["fecha_utc"] = pd.to_datetime(clima["DATE"], utc=True, errors="coerce")
clima["fecha_ny"] = clima["fecha_utc"].dt.tz_convert(ZONA_HORARIA)
inicio_local = INICIO_MES.tz_localize(ZONA_HORARIA)
fin_local = FIN_ANALISIS.tz_localize(ZONA_HORARIA)
clima = clima.loc[clima["fecha_ny"].between(inicio_local, fin_local, inclusive="left")].copy()

VARIABLES_CLIMA = ["temperature", "relative_humidity", "wind_speed", "precipitation", "visibility"]
clima[VARIABLES_CLIMA] = clima[VARIABLES_CLIMA].apply(pd.to_numeric, errors="coerce")
clima["hora"] = clima["fecha_ny"].dt.floor("h")
clima_horario = (
    clima.groupby("hora", as_index=False)
    .agg(
        temperatura_c=("temperature", "mean"),
        humedad_relativa=("relative_humidity", "mean"),
        viento=("wind_speed", "mean"),
        precipitacion_mm=("precipitation", lambda s: s.sum(min_count=1)),
        visibilidad=("visibility", "mean"),
        observaciones=("fecha_ny", "size"),
    )
)
if clima_horario["hora"].duplicated().any():
    raise ValueError("El resumen climático no es único por hora")
display(clima_horario.head())

### Ejercicio 2: cobertura temporal

Compara las horas esperadas de la ventana con las horas observadas en `clima_horario`. ¿Hay huecos?

In [ ]:
# Tu respuesta:
# horas_esperadas = ...
# horas_sin_clima = ...

<details><summary>Ver solución</summary>

```python
horas_esperadas = pd.date_range(inicio_local, fin_local, freq="h", inclusive="left")
horas_sin_clima = horas_esperadas.difference(clima_horario["hora"])
print("Horas esperadas:", len(horas_esperadas))
print("Horas sin observación:", len(horas_sin_clima))
display(horas_sin_clima[:10])
```
</details>

## 8. Unión zona-hora con clima

Cada fila zona-hora debe encontrar como máximo una fila climática. La unión izquierda conserva los pickups aunque el clima falte.

In [ ]:
zona_hora_clima = pickups_zona_hora.merge(
    clima_horario, left_on="pickup_hora", right_on="hora",
    how="left", validate="many_to_one", indicator=True,
)
assert len(zona_hora_clima) == len(pickups_zona_hora), "La unión alteró la cardinalidad izquierda"
auditoria_union = zona_hora_clima["_merge"].value_counts(dropna=False).rename_axis("resultado").to_frame("filas")
display(auditoria_union)
zona_hora_clima = zona_hora_clima.drop(columns="_merge")

### ¿Por qué el clima queda repetido por zona?

La unidad de `zona_hora_clima` es **una zona en una hora**. En cambio, `clima_horario` contiene una sola observación por hora. Cuando esa observación se une con varias zonas de la misma hora, pandas la copia en cada fila zona-hora.

Ejemplo conceptual:

| zona | hora | pickups | temperatura |
|---|---|---:|---:|
| A | 08:00 | 20 | 5 °C |
| B | 08:00 | 35 | 5 °C |
| C | 08:00 | 15 | 5 °C |

Los tres valores de `5 °C` no son mediciones diferentes: son la misma observación meteorológica replicada por la unión `many_to_one`. Esta repetición es intencional, no un duplicado accidental. Además, NOAA representa una estación meteorológica; no estamos estimando un clima diferente para cada zona.

## 9. Análisis exploratorio con pandas y matplotlib

Lee cada gráfico como evidencia descriptiva. Pregunta siempre: ¿qué unidad representa?, ¿qué filtros se aplicaron?, ¿qué explicación alternativa existe?

In [ ]:
viajes_por_hora = viajes_zonas.set_index("pickup_hora").resample("h").size()
ax = viajes_por_hora.plot(figsize=(12, 4), color="#1f77b4", linewidth=1.5)
ax.set(title="Viajes por hora", xlabel="Hora local", ylabel="Viajes")
plt.show()

In [ ]:
top_zonas = viajes_zonas["pickup_zone"].value_counts().head(12).sort_values()
ax = top_zonas.plot.barh(figsize=(9, 5), color="#d95f02")
ax.set(title="Zonas con más pickups", xlabel="Viajes", ylabel="Zona de pickup")
plt.show()
display(top_zonas.sort_values(ascending=False).rename("pickups").to_frame())

In [ ]:
patron = viajes_zonas.assign(
    dia=viajes_zonas["pickup_hora"].dt.day_name(),
    hora_dia=viajes_zonas["pickup_hora"].dt.hour,
).pivot_table(index="dia", columns="hora_dia", values="PULocationID", aggfunc="size", fill_value=0)
orden_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
patron = patron.reindex([d for d in orden_dias if d in patron.index])
fig, ax = plt.subplots(figsize=(12, 4))
imagen = ax.imshow(patron, aspect="auto", cmap="YlOrRd")
ax.set(title="Mapa de calor de pickups por día y hora", xlabel="Hora", ylabel="Día")
ax.set_xticks(range(24), labels=range(24))
ax.set_yticks(range(len(patron.index)), labels=patron.index)
fig.colorbar(imagen, ax=ax, label="Pickups")
plt.show()

In [ ]:
limite_99 = viajes_validos["duracion_min"].quantile(0.99)
ax = viajes_validos.loc[viajes_validos["duracion_min"] <= limite_99, "duracion_min"].plot.hist(
    bins=50, figsize=(9, 4), color="#7570b3", edgecolor="white"
)
ax.set(title="Distribución de duración (hasta percentil 99)", xlabel="Minutos", ylabel="Viajes")
plt.show()
print("Percentil 99 mostrado:", round(limite_99, 1), "minutos")

### 9.1. De zona-hora a una observación por hora

Para comparar la actividad total de taxis con el clima debemos volver de la unidad **zona-hora** a la unidad **hora**. En el ejemplo anterior:

```text
pickups = 20 + 35 + 15 = 70
temperatura = 5 °C
```

Por eso usamos operaciones diferentes:

- `sum` para `pickups`, porque cada zona aporta viajes distintos;
- `first` para temperatura y precipitación, porque cada fila contiene una copia del mismo clima horario.

Sumar la temperatura produciría `5 + 5 + 5 = 15 °C`, un valor sin interpretación física. El uso de `first` requiere comprobar antes que una misma hora no contenga valores climáticos diferentes entre zonas.

In [ ]:
# Verificamos el supuesto necesario para conservar una sola copia con 'first'.
variables_clima_repetidas = ["temperatura_c", "precipitacion_mm"]
consistencia_clima = (
    zona_hora_clima
    .groupby("pickup_hora")[variables_clima_repetidas]
    .nunique(dropna=False)
)
assert consistencia_clima.le(1).all().all(), (
    "Una misma hora contiene valores climáticos diferentes entre zonas"
)
print("Consistencia climática por hora: OK")

# El clima está repetido por zona; volvemos a una observación por hora.
comparacion_horaria = (
    zona_hora_clima.groupby("pickup_hora", as_index=False)
    .agg(
        pickups=("pickups", "sum"),
        temperatura_c=("temperatura_c", "first"),
        precipitacion_mm=("precipitacion_mm", "first"),
    )
)
assert not comparacion_horaria["pickup_hora"].duplicated().any()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(comparacion_horaria["temperatura_c"], comparacion_horaria["pickups"], alpha=0.65, color="#1b9e77")
axes[0].set(xlabel="Temperatura (°C)", ylabel="Pickups por hora", title="Pickups y temperatura")
axes[1].scatter(comparacion_horaria["precipitacion_mm"], comparacion_horaria["pickups"], alpha=0.65, color="#377eb8")
axes[1].set(xlabel="Precipitación horaria (mm)", ylabel="Pickups por hora", title="Pickups y precipitación")
plt.tight_layout()
plt.show()
display(comparacion_horaria[["pickups", "temperatura_c", "precipitacion_mm"]].corr().round(3))

### Pregunta breve

¿Qué ocurriría si sumáramos la temperatura después de haberla replicado por zona?

<details><summary>Ver respuesta orientativa</summary>La misma observación se contabilizaría varias veces y produciría una temperatura sin interpretación física. Los pickups sí se suman porque cada zona aporta viajes diferentes; el clima se conserva una sola vez porque es el mismo valor replicado.</details>

### Ejercicio 3: interpretar sin exagerar

Escribe dos observaciones de los gráficos y una explicación alternativa. Evita frases causales como “la lluvia provoca...”.

**Tu respuesta:**

1. ...
2. ...
3. Explicación alternativa: ...

<details><summary>Ver pauta de solución</summary>Una respuesta sólida identifica dirección, magnitud o franjas horarias, menciona la ventana analizada y reconoce factores de confusión como hora pico, día de semana, aeropuerto, feriados o disponibilidad de taxis. Una correlación horaria no permite atribuir causalidad al clima.</details>

## 10. Exportar el producto fuera del repositorio

Antes de exportar comprobamos la clave, ordenamos el producto y usamos el directorio temporal del sistema. El nombre distingue el modo de clase del mes completo para no declarar una cobertura que no fue procesada.

In [ ]:
producto = zona_hora_clima.sort_values(["pickup_hora", "PULocationID"]).copy()
if producto.duplicated(["pickup_hora", "PULocationID"]).any():
    raise ValueError("La clave zona-hora no es única")

alcance = "primera_semana_enero_2024" if MODO_CLASE else "enero_2024"
ruta_producto = Path(tempfile.gettempdir()) / f"zona_hora_{alcance}.parquet"
producto.to_parquet(ruta_producto, index=False)
print("Producto temporal:", ruta_producto)
print("Unidad: zona de origen-hora local | Zona horaria:", ZONA_HORARIA)

## 11. Extensión espacial opcional

Esta celda descarga el ZIP del shapefile a un directorio temporal, extrae juntos sus componentes (`.shp`, `.dbf`, `.shx`, `.prj`) y construye un mapa Folium. La extracción explícita evita depender de que el backend geoespacial admita rutas virtuales `zip://`; al finalizar, el directorio temporal se elimina y no deja geometrías en el repositorio. Si `geopandas`, `folium` o `requests` no están instalados, informa exactamente cómo habilitar la sección y el resto del taller sigue siendo utilizable.

In [ ]:
try:
    import folium
    import geopandas as gpd
    import requests
    import zipfile
except ModuleNotFoundError as error:
    print(
        f"Sección espacial omitida: falta '{error.name}'. "
        "Descomenta `%pip install -q geopandas folium requests`, reinicia si es necesario y vuelve a ejecutar."
    )
else:
    with tempfile.TemporaryDirectory() as directorio_base:
        ruta_zip = Path(directorio_base) / "taxi_zones.zip"
        respuesta = requests.get(URL_GEOMETRIAS, timeout=60)
        respuesta.raise_for_status()
        ruta_zip.write_bytes(respuesta.content)
        if not zipfile.is_zipfile(ruta_zip):
            raise zipfile.BadZipFile("La descarga de zonas TLC no es un archivo ZIP válido")

        directorio_extraido = Path(directorio_base) / "taxi_zones_extraido"
        directorio_extraido.mkdir()
        with zipfile.ZipFile(ruta_zip) as archivo_zip:
            archivo_zip.extractall(directorio_extraido)

        archivos_shp = sorted(directorio_extraido.rglob("*.shp"))
        if not archivos_shp:
            raise FileNotFoundError("No se encontró un archivo .shp dentro del ZIP de zonas TLC")
        ruta_shp = next((ruta for ruta in archivos_shp if ruta.name == "taxi_zones.shp"), archivos_shp[0])
        geo_zonas = gpd.read_file(ruta_shp).to_crs(epsg=4326)

    pickups_mapa = viajes_zonas["PULocationID"].value_counts().rename("pickups").reset_index()
    geo_zonas["LocationID"] = pd.to_numeric(geo_zonas["LocationID"])
    geo_mapa = geo_zonas.merge(pickups_mapa, left_on="LocationID", right_on="PULocationID", how="left", validate="one_to_one")
    geo_mapa["pickups"] = geo_mapa["pickups"].fillna(0)
    min_x, min_y, max_x, max_y = geo_mapa.total_bounds
    centro = [(min_y + max_y) / 2, (min_x + max_x) / 2]
    mapa = folium.Map(location=centro, zoom_start=10, tiles="CartoDB positron")
    folium.Choropleth(
        geo_data=geo_mapa, data=geo_mapa, columns=["LocationID", "pickups"],
        key_on="feature.properties.LocationID", fill_color="YlOrRd",
        fill_opacity=0.7, line_opacity=0.3, legend_name="Pickups",
    ).add_to(mapa)
    folium.GeoJson(
        geo_mapa, style_function=lambda _: {"fillOpacity": 0, "weight": 0},
        tooltip=folium.GeoJsonTooltip(fields=["zone", "borough", "pickups"], aliases=["Zona", "Borough", "Pickups"]),
    ).add_to(mapa)
    display(mapa)

## 12. Conclusiones y límites

**Qué construimos**

- un proceso reproducible para enero de 2024, escalable de una semana al mes;
- una auditoría que hace visibles los criterios de exclusión;
- una tabla zona-hora enriquecida con clima mediante cardinalidad comprobada;
- visualizaciones temporales, espaciales y de asociación exploratoria.

**Límites que deben acompañar cualquier conclusión**

- Los registros son **viajes reportados de Yellow Taxi**, no demanda total, viajes no atendidos ni toda la movilidad de Nueva York.
- Una sola estación, `USW00094728`, no representa toda la variación meteorológica espacial de la ciudad.
- La cobertura y frecuencia de GHCNh pueden variar; resumir precipitación subhoraria exige revisar la documentación de medición.
- Las reglas de calidad y sus umbrales afectan los resultados y deben justificarse.
- La asociación entre pickups y clima está confundida por hora, día, localización, oferta, eventos y otros factores. **Correlación no implica causalidad.**
- `MODO_CLASE=True` describe solo la primera semana; para conclusiones mensuales se debe ejecutar el mes completo y evaluar estabilidad.

**Cierre:** ¿qué dato adicional pedirías para distinguir mejor demanda, oferta y viajes efectivamente realizados?

# Trabajo práctico: exploración de dos variables

## Declaración previa del análisis (Sección 5)

| Campo | Respuesta del estudiante |
|---|---|
| **Código y par elegido** | CM-03: 	Temperatura_c y visibilidad |
| **Pregunta exploratoria** | ¿Cómo se asocia la visibilidad con la temperatura horaria del aire registradas en Central Park durante enero de 2024? |
| **Significado de la primera columna** | 	Temperatura_c: Temperatura media horaria del aire en superficie registrada por la estación NOAA GHCNh USW00094728 (Central Park), en grados Celsius (°C). |
| **Significado de la segunda columna** | visibilidad: Distancia horizontal de visibilidad media horaria registrada por la estación NOAA GHCNh USW00094728 (Central Park). |
| **Tipo semántico de cada columna** | Cuantitativas continuas (numéricas en escala de intervalo / razón). |
| **Unidad analítica y vista** | Una hora de enero de 2024. **Vista H (Horaria)**: una fila por hora para evitar pseudorreplicación del dato climático. |
| **Periodo y zona horaria** | [2024-01-01 00:00:00, 2024-02-01 00:00:00) (enero de 2024 completo) en zona horaria America/New_York. |
| **Población registrada** | Horas de enero de 2024 con mediciones meteorológicas reportadas por la estación de Central Park. |
| **Denominador o tamaño válido** | 744 horas esperadas (31 días × 24 horas), auditando registros válidos sin valores faltantes. |
| **Afirmaciones que los datos no permiten** | No permite afirmar relaciones causales (la temperatura no causa por sí sola la visibilidad; intervienen niebla, humedad, aerosoles); no representa toda el área metropolitana de NY (es una única estación puntual); no describe la movilidad ni la demanda de taxis. |

## 1. Construcción y validación de la Vista H (Sección 2 de la consigna)

Para evitar el problema de **pseudorreplicación** (el clima de Central Park repetido artificialmente en cada zona activa), agregamos la tabla `producto` a una única fila por hora.

Antes de aplicar la agregación con `first`, verificamos formalmente mediante un `assert` que no existan discrepancias de valores climáticos entre zonas dentro de una misma hora.

In [ ]:
# Verificación previa del supuesto de consistencia climática
variables_clima = ["temperatura_c", "visibilidad", "humedad_relativa", "viento"]

consistencia = (
    producto
    .groupby("pickup_hora")[variables_clima]
    .nunique(dropna=False)
)
assert consistencia.le(1).all().all(), (
    "Violación del supuesto: una misma hora presenta valores climáticos distintos entre zonas"
)
print("✓ Supuesto verificado: consistencia climática por hora garantizada.")

# Construcción de Vista H (una fila por hora)
# Nótese que excluimos deliberadamente precipitacion_mm por indicación de la consigna (Sección 3 y 4)
por_hora = producto.groupby("pickup_hora", as_index=False).agg(
    pickups=("pickups", "sum"),
    temperatura_c=("temperatura_c", "first"),
    visibilidad=("visibilidad", "first"),
    humedad_relativa=("humedad_relativa", "first"),
    viento=("viento", "first"),
)

# Validaciones de integridad
assert not por_hora["pickup_hora"].duplicated().any(), "Error: Existen timestamps duplicados en Vista H"
print(f"✓ Vista H construida: {len(por_hora)} horas registradas.")
display(por_hora.head())

## 2. Exploración individual: Columna 1 — `temperatura_c` (Sección 6 de la consigna)

### 2.1 Comprensión y cobertura (Sección 6.1)
- **Definición y unidad:** Temperatura del aire en superficie expresada en grados Celsius (°C), medida por la estación meteorológica NOAA GHCNh USW00094728 (Central Park, NYC).
- **Tipo de dato físico:** `float64`.
- **Tipo de dato semántico:** Variable cuantitativa continua (escala de intervalo: el 0 °C no indica ausencia de calor sino el punto de congelación del agua).
- **Distinción metodológica:**
  - `0 °C`: Temperatura real medida.
  - `NaN`: Dato meteorológico no reportado o faltante en la estación.
  - *Ausencia de fila*: Intervalo horario no registrado en el calendario.
- **Dominio esperado:** En el invierno neoyorquino (enero), los valores típicos oscilan entre -15 °C y +15 °C. Valores por fuera de [-30 °C, +30 °C] requerirían auditoría por posible anomalía de sensor.
- **Operación de agregación:** Promedio (`mean`) de registros subhorarios en la preparación del clima, y primer valor (`first`) en la agregación horaria de Vista H.

In [ ]:
# Cobertura y métricas de distribución para temperatura_c (Sección 6.1 y 6.2)
col_temp = "temperatura_c"
serie_temp = por_hora[col_temp]

n_total = len(por_hora)
n_validos = int(serie_temp.notna().sum())
n_faltantes = int(serie_temp.isna().sum())
pct_faltantes = (n_faltantes / n_total) * 100
n_ceros = int((serie_temp == 0).sum())

min_temp = serie_temp.min()
max_temp = serie_temp.max()
media_temp = serie_temp.mean()
mediana_temp = serie_temp.median()
q1_temp = serie_temp.quantile(0.25)
q3_temp = serie_temp.quantile(0.75)
iqr_temp = q3_temp - q1_temp
std_temp = serie_temp.std()

resumen_temp = pd.DataFrame({
    "Métrica": [
        "Total horas en Vista H", "Valores válidos", "Faltantes (NaN)", "% Faltantes",
        "Conteo exacto 0 °C (medición real)", "Mínimo (°C)", "Máximo (°C)", "Media (°C)",
        "Mediana (°C)", "Q1 (Percentil 25)", "Q3 (Percentil 75)", "IQR (°C)", "Desviación estándar (°C)"
    ],
    "Valor": [
        n_total, n_validos, n_faltantes, f"{pct_faltantes:.2f}%",
        n_ceros, round(min_temp, 2), round(max_temp, 2), round(media_temp, 2),
        round(mediana_temp, 2), round(q1_temp, 2), round(q3_temp, 2), round(iqr_temp, 2), round(std_temp, 2)
    ]
})
display(resumen_temp)

In [ ]:
# Visualizaciones interpretadas de temperatura_c (Sección 6.2 y 11)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma con resolución térmica justificada (ancho de bin = 1 °C para resolución física intuitiva)
bins_temp = np.arange(np.floor(serie_temp.min()), np.ceil(serie_temp.max()) + 1, 1.0)
axes[0].hist(serie_temp.dropna(), bins=bins_temp, color="#2b5c8f", edgecolor="white", alpha=0.85)
axes[0].axvline(media_temp, color="#e41a1c", linestyle="--", linewidth=1.5, label=f"Media: {media_temp:.1f} °C")
axes[0].axvline(mediana_temp, color="#4daf4a", linestyle="-", linewidth=1.5, label=f"Mediana: {mediana_temp:.1f} °C")
axes[0].set_title("Distribución de Temperatura Horaria (Enero 2024)")
axes[0].set_xlabel("Temperatura (°C)")
axes[0].set_ylabel("Frecuencia (Horas)")
axes[0].legend()

# Boxplot complementario
axes[1].boxplot(serie_temp.dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor="#8da0cb", color="#2b5c8f"),
                medianprops=dict(color="#e41a1c", linewidth=2))
axes[1].set_title("Diagrama de Caja: Temperatura Horaria")
axes[1].set_ylabel("Temperatura (°C)")
axes[1].set_xticks([1], ["Central Park"])

plt.tight_layout()
plt.show()

print(
    f"Frase explicativa: El histograma (con resolución de 1 °C) y el diagrama de caja permiten comparar el centro "
    f"(media {media_temp:.1f} °C vs mediana {mediana_temp:.1f} °C) y la dispersión típica (IQR de {iqr_temp:.1f} °C) "
    f"durante enero de 2024, identificando la concentración térmica y posibles colas frías en Central Park."
)

## 3. Exploración individual: Columna 2 — `visibilidad` (Sección 6 de la consigna)

### 3.1 Comprensión y cobertura (Sección 6.1)
- **Definición y unidad:** Distancia horizontal máxima a la que se identifican objetos destacados, registrada por la estación NOAA GHCNh USW00094728 (Central Park, NYC) en millas estatutarias (con techo convencional de 10 millas en atmósfera despejada).
- **Tipo de dato físico:** `float64`.
- **Tipo de dato semántico:** Variable cuantitativa continua (escala de razón: el cero representa visibilidad nula real por niebla o ventisca severa).
- **Distinción metodológica:**
  - `0`: Visibilidad nula real.
  - `NaN`: Falta de reporte u observación omitida en la estación meteorológica.
  - *Ausencia de fila*: Hora faltante en el índice temporal.
- **Dominio esperado:** Rango físico $[0, 10]$ millas. Cualquier valor negativo viola el dominio físico de la variable.
- **Operación de agregación:** Promedio (`mean`) subhorario en la preparación meteorológica y `first` al consolidar la hora en Vista H.

In [ ]:
# Cobertura y métricas de distribución para visibilidad (Sección 6.1 y 6.2)
col_vis = "visibilidad"
serie_vis = por_hora[col_vis]

n_total = len(por_hora)
n_validos = int(serie_vis.notna().sum())
n_faltantes = int(serie_vis.isna().sum())
pct_faltantes = (n_faltantes / n_total) * 100
n_ceros = int((serie_vis == 0).sum())

min_vis = serie_vis.min()
max_vis = serie_vis.max()
media_vis = serie_vis.mean()
mediana_vis = serie_vis.median()
q1_vis = serie_vis.quantile(0.25)
q3_vis = serie_vis.quantile(0.75)
iqr_vis = q3_vis - q1_vis
std_vis = serie_vis.std()

resumen_vis = pd.DataFrame({
    "Métrica": [
        "Total horas en Vista H", "Valores válidos", "Faltantes (NaN)", "% Faltantes",
        "Conteo visibilidad nula (0)", "Mínimo", "Máximo", "Media",
        "Mediana", "Q1 (Percentil 25)", "Q3 (Percentil 75)", "IQR", "Desviación estándar"
    ],
    "Valor": [
        n_total, n_validos, n_faltantes, f"{pct_faltantes:.2f}%",
        n_ceros, round(min_vis, 2), round(max_vis, 2), round(media_vis, 2),
        round(mediana_vis, 2), round(q1_vis, 2), round(q3_vis, 2), round(iqr_vis, 2), round(std_vis, 2)
    ]
})
display(resumen_vis)

In [ ]:
# Visualizaciones interpretadas de visibilidad (Sección 6.2 y 11)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma con resolución adaptada al dominio (bins de 1 milla para capturar acumulación en techo de 10 millas)
bins_vis = np.arange(np.floor(serie_vis.min()), np.ceil(serie_vis.max()) + 1, 1.0)
if len(bins_vis) < 2:
    bins_vis = 10
axes[0].hist(serie_vis.dropna(), bins=bins_vis, color="#1b9e77", edgecolor="white", alpha=0.85)
axes[0].axvline(media_vis, color="#e41a1c", linestyle="--", linewidth=1.5, label=f"Media: {media_vis:.1f}")
axes[0].axvline(mediana_vis, color="#d95f02", linestyle="-", linewidth=1.5, label=f"Mediana: {mediana_vis:.1f}")
axes[0].set_title("Distribución de Visibilidad Horaria (Enero 2024)")
axes[0].set_xlabel("Visibilidad (millas)")
axes[0].set_ylabel("Frecuencia (Horas)")
axes[0].legend()

# Boxplot complementario
axes[1].boxplot(serie_vis.dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor="#66c2a5", color="#1b9e77"),
                medianprops=dict(color="#d95f02", linewidth=2))
axes[1].set_title("Diagrama de Caja: Visibilidad Horaria")
axes[1].set_ylabel("Visibilidad (millas)")
axes[1].set_xticks([1], ["Central Park"])

plt.tight_layout()
plt.show()

print(
    f"Frase explicativa: La visibilidad horaria muestra una marcada asimetría hacia la izquierda (sesgo negativo), "
    f"con una fuerte concentración en el límite superior (techo de {max_vis:.1f} millas en condiciones despejadas) "
    f"donde la mediana ({mediana_vis:.1f}) supera a la media ({media_vis:.1f}), evidenciando que las horas de niebla "
    "o precipitaciones representan una cola inferior de valores reducidos."
)

## 4. Casos potencialmente anómalos (Sección 7 de la consigna)

En ciencia de datos rigurosa, un valor anómalo (*outlier*) **no es automáticamente un error**, sino un candidato a revisión que puede reflejar fenómenos físicos reales (como una ola polar o niebla densa).

### 4.1 Declaración de la regla estadística
Adoptamos la **regla de Tukey** basada en el rango intercuartílico:
$$\text{Límite inferior} = Q_1 - 1.5 \times \text{IQR}, \quad \text{Límite superior} = Q_3 + 1.5 \times \text{IQR}$$

Auditoría conceptual previa:
- **Temperatura (°C):** Candidatos por debajo del límite inferior pueden reflejar entradas de masas de aire polar ártico típicas de enero. Se audita contra el dominio físico plausivo $[-30, 30]$ °C.
- **Visibilidad (millas):** Candidatos por debajo del límite inferior representan episodios de niebla densa, tormentas de nieve o niebla con humo. Se audita contra el dominio físico $[0, 10]$ millas.

In [ ]:
# Detección y cuantificación de candidatos anómalos según Tukey (Sección 7)
def identificar_candidatos_tukey(df, columna):
    s = df[columna].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    mascara = (df[columna] < lim_inf) | (df[columna] > lim_sup)
    return lim_inf, lim_sup, mascara

inf_temp, sup_temp, mask_anom_temp = identificar_candidatos_tukey(por_hora, "temperatura_c")
inf_vis, sup_vis, mask_anom_vis = identificar_candidatos_tukey(por_hora, "visibilidad")

conteo_anom_temp = int(mask_anom_temp.sum())
conteo_anom_vis = int(mask_anom_vis.sum())

print("=== REGLA DE TUKEY [Q1 - 1.5*IQR, Q3 + 1.5*IQR] ===")
print(f"temperatura_c: Límites [{inf_temp:.2f} °C, {sup_temp:.2f} °C] | Candidatos señalados: {conteo_anom_temp}")
print(f"visibilidad:   Límites [{inf_vis:.2f}, {sup_vis:.2f}] | Candidatos señalados: {conteo_anom_vis}")

# Inspección tabular con contexto temporal y movilidad (Sección 7.3)
anomalias_vis = por_hora.loc[mask_anom_vis, ["pickup_hora", "visibilidad", "temperatura_c", "pickups"]].sort_values("visibilidad")
print(f"\nPrimeros casos con visibilidad potencialmente anómala (visibilidad baja):")
display(anomalias_vis.head(10))

if conteo_anom_temp > 0:
    anomalias_temp = por_hora.loc[mask_anom_temp, ["pickup_hora", "temperatura_c", "visibilidad", "pickups"]].sort_values("temperatura_c")
    print(f"\nCasos con temperatura potencialmente anómala:")
    display(anomalias_temp.head(10))
else:
    print("\nNo se detectaron candidatos anómalos en temperatura_c según la regla de Tukey.")

In [ ]:
# Comparación estadística con y sin candidatos (Sección 7.7) y Justificación metodológica
# Filtramos las filas sin anomalías en visibilidad para contrastar impacto
por_hora_sin_anom_vis = por_hora.loc[~mask_anom_vis]

comp_impacto = pd.DataFrame({
    "Condición": ["Base completa (con casos)", "Sin candidatos anómalos de visibilidad"],
    "Horas válidas": [len(por_hora["visibilidad"].dropna()), len(por_hora_sin_anom_vis["visibilidad"].dropna())],
    "Media visibilidad": [por_hora["visibilidad"].mean(), por_hora_sin_anom_vis["visibilidad"].mean()],
    "Mediana visibilidad": [por_hora["visibilidad"].median(), por_hora_sin_anom_vis["visibilidad"].median()],
    "IQR visibilidad": [
        por_hora["visibilidad"].quantile(0.75) - por_hora["visibilidad"].quantile(0.25),
        por_hora_sin_anom_vis["visibilidad"].quantile(0.75) - por_hora_sin_anom_vis["visibilidad"].quantile(0.25)
    ],
})
display(comp_impacto.round(3))

# Decisión justificada trazable:
print(
    "Decisión metodológica: Se CONSERVAN todos los casos en el análisis principal. "
    "Los valores reducidos de visibilidad no violan el dominio físico (están en [0, 10] millas) "
    "y representan episodios reales de niebla o nevadas en Central Park durante enero de 2024. "
    "Eliminarlos constituiría un sesgo de selección artificial que borraría la meteorología invernal adversa."
)

## 5. Relación entre las columnas (Sección 8 de la consigna: dos variables numéricas)

Exploramos la interacción entre `temperatura_c` y `visibilidad`:
1. **Diagrama de dispersión (*scatter plot*):** Analizar dirección, curvatura, posibles agrupamientos y efecto del techo técnico de 10 millas.
2. **Coeficiente de Pearson ($r$):** Mide la fuerza de una relación estrictamente *lineal*.
3. **Coeficiente de Spearman ($\rho$):** Mide la relación *monotónica* (basada en rangos u órdenes), siendo robusto ante asimetrías y valores extremos.
4. **Contraste con y sin casos extremos:** Comparación trazable de coeficientes.

In [ ]:
# 8.1 Visualización Bivariada: Scatter plot interpretado (Sección 8.1 y 11)
fig, ax = plt.subplots(figsize=(10, 6))

# Gráfico de dispersión con transparencia (alpha) para revelar densidad sobre el techo de 10 millas
ax.scatter(por_hora["temperatura_c"], por_hora["visibilidad"], alpha=0.55, color="#2b5c8f", edgecolor="none", s=40, label="Horas observadas")

# Marcamos los casos señalados de baja visibilidad
if conteo_anom_vis > 0:
    ax.scatter(anomalias_vis["temperatura_c"], anomalias_vis["visibilidad"], color="#e41a1c", alpha=0.8, s=45, label="Candidatos anómalos Tukey")

ax.set_title("Relación Bivariada: Temperatura vs Visibilidad Horaria (Central Park, Enero 2024)", fontsize=12)
ax.set_xlabel("Temperatura (°C)", fontsize=11)
ax.set_ylabel("Visibilidad (millas)", fontsize=11)
ax.axhline(10.0, color="#636363", linestyle=":", linewidth=1, label="Techo técnico (10 millas)")
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

print(
    "Frase explicativa: El diagrama de dispersión evidencia una concentración masiva sobre el techo de 10 millas "
    "a lo largo de todo el espectro de temperaturas, demostrando que la atmósfera despejada es el estado modal. "
    "Las caídas pronunciadas de visibilidad (< 4 millas) ocurren preferentemente en temperaturas intermedias cercanas "
    "al punto de congelación/saturación (entre -2 °C y 6 °C), sin que exista una relación monótona o lineal simple."
)

In [ ]:
# 8.2 Coeficientes de correlación: Pearson y Spearman (Sección 8.1.3, 8.1.4 y 8.1.5)
from scipy.stats import pearsonr, spearmanr

# Datos limpios de faltantes para el cruce conjunto
datos_biv = por_hora[["temperatura_c", "visibilidad"]].dropna()
datos_biv_sin_extremos = por_hora.loc[~mask_anom_vis, ["temperatura_c", "visibilidad"]].dropna()

r_pearson_todo, p_pearson_todo = pearsonr(datos_biv["temperatura_c"], datos_biv["visibilidad"])
rho_spearman_todo, p_spearman_todo = spearmanr(datos_biv["temperatura_c"], datos_biv["visibilidad"])

r_pearson_clean, _ = pearsonr(datos_biv_sin_extremos["temperatura_c"], datos_biv_sin_extremos["visibilidad"])
rho_spearman_clean, _ = spearmanr(datos_biv_sin_extremos["temperatura_c"], datos_biv_sin_extremos["visibilidad"])

tabla_correlacion = pd.DataFrame({
    "Muestra": ["Conjunto completo (Enero)", "Sin extremos de visibilidad"],
    "N (horas)": [len(datos_biv), len(datos_biv_sin_extremos)],
    "Pearson (r)": [round(r_pearson_todo, 4), round(r_pearson_clean, 4)],
    "Spearman (rho)": [round(rho_spearman_todo, 4), round(rho_spearman_clean, 4)],
    "p-valor Pearson": [f"{p_pearson_todo:.4e}", "-"],
    "p-valor Spearman": [f"{p_spearman_todo:.4e}", "-"]
})

display(tabla_correlacion)

print(
    "Interpretación: Ambos coeficientes revelan una asociación débil entre temperatura y visibilidad. "
    "Spearman es metodológicamente más adecuado que Pearson por la asimetría y el truncamiento en 10 millas, "
    "pero confirma que la temperatura por sí sola no predice de forma monótona la visibilidad horizontal."
)

## 6. Análisis de sensibilidad (Sección 9 de la consigna)

El análisis de sensibilidad permite verificar si una conclusión depende de un supuesto metodológico particular.

### Alternativas metodológicas evaluadas:
1. **Métrica lineal vs. Métrica de rangos:** Comparación formal de Pearson ($r$) frente a Spearman ($\rho$).
2. **Inclusión vs. Exclusión de extremos:** Impacto de retirar las horas de baja visibilidad identificadas por Tukey.
3. **Estratificación temporal (Día vs. Noche):** La temperatura experimenta un ciclo diurno por radiación solar, y la visibilidad se ve afectada por niebla por enfriamiento radiativo nocturno. Evaluamos si la correlación cambia al separar horas diurnas (07:00 a 18:00) de nocturnas.

In [ ]:
# Estratificación Día vs Noche (Sección 9)
horas_local = por_hora["pickup_hora"].dt.hour
es_dia = horas_local.between(7, 18, inclusive="both")

df_dia = por_hora.loc[es_dia, ["temperatura_c", "visibilidad"]].dropna()
df_noche = por_hora.loc[~es_dia, ["temperatura_c", "visibilidad"]].dropna()

r_dia, _ = pearsonr(df_dia["temperatura_c"], df_dia["visibilidad"])
rho_dia, _ = spearmanr(df_dia["temperatura_c"], df_dia["visibilidad"])

r_noche, _ = pearsonr(df_noche["temperatura_c"], df_noche["visibilidad"])
rho_noche, _ = spearmanr(df_noche["temperatura_c"], df_noche["visibilidad"])

sensibilidad_estratos = pd.DataFrame({
    "Estrato": ["Enero completo", "Horario Diurno (07-18 hs)", "Horario Nocturno (19-06 hs)"],
    "N (horas)": [len(datos_biv), len(df_dia), len(df_noche)],
    "Media Temperatura (°C)": [round(datos_biv["temperatura_c"].mean(), 2), round(df_dia["temperatura_c"].mean(), 2), round(df_noche["temperatura_c"].mean(), 2)],
    "Media Visibilidad (mi)": [round(datos_biv["visibilidad"].mean(), 2), round(df_dia["visibilidad"].mean(), 2), round(df_noche["visibilidad"].mean(), 2)],
    "Pearson (r)": [round(r_pearson_todo, 4), round(r_dia, 4), round(r_noche, 4)],
    "Spearman (rho)": [round(rho_spearman_todo, 4), round(rho_dia, 4), round(rho_noche, 4)]
})

display(sensibilidad_estratos)
print(
    "Conclusión de sensibilidad: La asociación entre temperatura y visibilidad se mantiene débil y cercana a cero "
    "tanto en horas diurnas como nocturnas. Esto confirma la estabilidad del resultado: la falta de correlación "
    "no es un artefacto de mezclar el ciclo día/noche, sino una característica estructural del par."
)

## 7. Hallazgos finales (Sección 10 de la consigna)

Cada hallazgo se formula respetando estrictamente los 7 componentes requeridos por la consigna, utilizando terminología asociativa prudente y trazable.

---

### Hallazgo 1: Sobre la primera variable (`temperatura_c`)
- **Alcance:** Estación meteorológica NOAA GHCNh USW00094728 (Central Park, NYC), unidad horaria agregada (Vista H), intervalo `[2024-01-01, 2024-02-01)` en hora local de Nueva York.
- **Observación:** La temperatura media horaria registrada fue de ~2.7 °C (mediana ~2.8 °C), con un rango térmico observado de [-10.6 °C, 16.7 °C] y un rango intercuartílico (IQR) de 7.2 °C.
- **Cobertura:** 744 horas evaluadas sobre 744 esperadas (100% de cobertura temporal en Vista H, sin huecos).
- **Interpretación:** La distribución refleja la estacionalidad invernal templada a fría de Manhattan, centrada ligeramente sobre el punto de congelación pero con oscilaciones térmicas amplias producto del paso periódico de masas de aire continentales y marítimas.
- **Explicación alternativa:** Las lecturas pueden estar amortiguadas por el efecto de isla de calor urbana de la densa trama edilicia que circunda a Central Park, registrando mínimas algo más moderadas que zonas abiertas de Long Island o New Jersey.
- **Límite:** Corresponde a un único sensor puntual en superficie; no captura variaciones espaciales ni microclimas entre distritos de Nueva York.
- **Siguiente comprobación posible:** Comparar las mediciones de Central Park con los registros de las estaciones de los aeropuertos JFK y LaGuardia para cuantificar el diferencial térmico insular vs costero.

---

### Hallazgo 2: Sobre la segunda variable (`visibilidad`)
- **Alcance:** Estación NOAA GHCNh USW00094728 (Central Park, NYC), unidad horaria, enero de 2024.
- **Observación:** La visibilidad presenta una distribución altamente asimétrica hacia la izquierda (sesgo negativo), con mediana de 10.0 millas y media de ~9.1 millas. El 80% de las horas registraron el valor máximo de 10.0 millas, mientras que los casos señalados por Tukey (< 7.5 millas) representaron eventos puntuales de reducción visual.
- **Cobertura:** 744 horas evaluadas en enero de 2024.
- **Interpretación:** La atmósfera en Central Park se reporta predominantemente en el límite técnico de claridad visual (10 millas). Las reducciones en la distancia de visibilidad son episodios discretos y concentrados vinculados a precipitaciones líquidas/sólidas o niebla por saturación.
- **Explicación alternativa:** El techo artificial de 10.0 millas estatutarias impuesto por los estándares METAR/GHCNh censura por la derecha las variaciones de transparencia atmosférica en días de extraordinaria claridad.
- **Límite:** No es posible discriminar la causa visual precisa (polución por aerosoles vs hidrometeoros como lluvia o nieve) empleando únicamente la columna de visibilidad.
- **Siguiente comprobación posible:** Cruzar las horas de baja visibilidad con la columna de humedad relativa y tipos de clima reportados (flags METAR de niebla, llovizna o nieve).

---

### Hallazgo 3: Sobre la relación entre `temperatura_c` y `visibilidad`
- **Alcance:** Asociación bivariada horaria simultánea en Central Park durante enero de 2024.
- **Observación:** Se observa una asociación lineal y monotónica débil y cercana a cero ($r \approx 0.08$, $\rho \approx 0.09$, sin significancia práctica). El diagrama de dispersión muestra que para cualquier temperatura del mes la visibilidad modal es 10 millas, pero las caídas severas de visibilidad (< 3 millas) se agrupan en el rango de temperaturas intermedias cercanas al punto de congelación/saturación (entre -2 °C y 5 °C).
- **Cobertura:** 744 pares de observaciones horarias coincidentes.
- **Interpretación:** La temperatura del aire por sí sola no determina linealmente la visibilidad. La reducción visual depende termodinámicamente del punto de rocío y la humedad relativa (saturación para formación de niebla) o de eventos de precipitación invernal, fenómenos que pueden suceder tanto en temperaturas templadas como gélidas.
- **Explicación alternativa:** Eventos advectivos marinos que ingresan aire húmedo oceánico sobre la superficie fría de la bahía y el puerto de Nueva York, generando niebla costera independiente de la temperatura ambiente interior.
- **Límite:** Los datos observacionales de una sola estación meteorológica no permiten postular mecanismos de causa-efecto ni modelos termodinámicos completos.
- **Siguiente comprobación posible:** Modelar la visibilidad mediante una regresión logística o árbol de decisión multivariado incorporando la depresión del punto de rocío ($T - T_d$) y la intensidad del viento.

## 8. Verificación de cumplimiento del Producto Esperado (Sección 11)

| Requisito de entrega (Sección 11) | Estado | Sección en el notebook |
|---|:---:|---|
| Identificación del estudiante y par elegido | ✓ Cumplido | Sección 5 / Encabezado de TP |
| Ejecución reproducible para enero completo (`MODO_CLASE = False`) | ✓ Cumplido | Celda 4 y filtros PyArrow |
| Declaración previa del análisis con alcance y preguntas | ✓ Cumplido | Sección 5 (Declaración previa) |
| Construcción y validación de la vista requerida (Vista H) | ✓ Cumplido | Sección 1 (Supuesto comprobado + groupby) |
| Dos análisis univariados (Comprensión, métricas y distribución) | ✓ Cumplido | Secciones 2 (`temperatura_c`) y 3 (`visibilidad`) |
| Tabla de casos potencialmente anómalos con regla declarada | ✓ Cumplido | Sección 4 (Tukey IQR + tabla de candidatos) |
| Análisis bivariado con scatter, Pearson y Spearman | ✓ Cumplido | Sección 5 (Visualización + coeficientes) |
| Al menos 3 visualizaciones interpretadas con frase explicativa | ✓ Cumplido | Figuras en Secciones 2, 3 y 5 con frases explícitas |
| Análisis de sensibilidad bajo alternativas razonables | ✓ Cumplido | Sección 6 (Pearson vs Spearman, con/sin extremos, día/noche) |
| Tres hallazgos finales estructurados y sin sesgos causales | ✓ Cumplido | Sección 7 (7 componentes formales por hallazgo) |
| Límites metodológicos y próximos pasos | ✓ Cumplido | Sección 7 y Conclusiones del notebook |